# Ручная проверка API поиска

Ноутбук отправляет HTTP-запрос в локальный FastAPI и показывает результаты поиска. Сначала запустите сервис из корня проекта: `python -m uvicorn src.api.main:app --reload`. Swagger доступен по адресу `http://127.0.0.1:8000/docs`. Для получения результатов нужен утверждённый корпус `data/knowledge/articles.jsonl`; без него API вернёт 503 и продолжит показывать статус базы знаний.


In [ ]:
import json
from urllib.error import HTTPError, URLError
from urllib.request import Request, urlopen

API_ROOT = 'http://127.0.0.1:8000'


In [ ]:
try:
    with urlopen(f'{API_ROOT}/health/live', timeout=5) as response:
        print('API:', json.loads(response.read().decode('utf-8')))
except URLError as error:
    print('Не удалось подключиться к API. Запустите uvicorn из корня проекта.')
    print(error)


In [ ]:
try:
    with urlopen(f'{API_ROOT}/health/knowledge/ready', timeout=5) as response:
        print('База знаний:', json.loads(response.read().decode('utf-8')))
except HTTPError as error:
    body = json.loads(error.read().decode('utf-8'))
    print(f'База знаний пока не готова: HTTP {error.code}')
    print(body)
except URLError as error:
    print('API недоступен. Сначала проверьте предыдущую ячейку.')
    print(error)


## Отправка поискового запроса

Измените запрос на свой. `top_k` ограничен API значением 10. Фильтр `language` можно оставить пустым или указать код языка статей.


In [ ]:
payload = {
    'text': 'card payment reversed',
    'top_k': 5,
    'language': 'en',
}
request = Request(
    f'{API_ROOT}/v1/knowledge/search',
    data=json.dumps(payload).encode('utf-8'),
    headers={'Content-Type': 'application/json'},
    method='POST',
)
try:
    with urlopen(request, timeout=15) as response:
        result = json.loads(response.read().decode('utf-8'))
    print('Request ID:', result['request_id'])
    print('Ранжировщик:', result['retriever'])
    for position, hit in enumerate(result['hits'], start=1):
        print(f"{position}. {hit['title']}")
        print(f"   Статья: {hit['document_id']} | фрагмент: {hit['chunk_id']}")
        print(f"   Источник: {hit['source']} | актуальность: {hit['updated_at']}")
        print(f"   Score: {hit['score']:.4f} ({hit['score_type']})")
        print(hit['text'])
except HTTPError as error:
    body = json.loads(error.read().decode('utf-8'))
    print(f'Поиск не выполнен: HTTP {error.code}')
    print(body)
except URLError as error:
    print('Не удалось подключиться к API.')
    print(error)


## Как оценивать выдачу

Проверьте, ведёт ли `source` к актуальной одобренной статье и действительно ли фрагмент подтверждает нужный ответ. `score` пригоден для сортировки только внутри выбранного ранжировщика; это не confidence. Ноутбук показывает сырую выдачу и не генерирует ответ клиенту. Для оценки качества используйте размеченную выборку и notebook `02_retrieval_exploration.ipynb`.
